# Policy Rate Monitor over HTTP

This notebook uses the tool only through its HTTP API, as a web page or another program would: it never imports the package.

> ⚠️ **Start the service first**  

> In a terminal, from the repository root, start the API and leave it running:
> 
>
```bash
uv sync --all-groups --all-extras      # once: the API needs the web extra
uv run bis-prates serve --out /tmp/bis-prates-out
```

> `--out` is where the report job at the end of this notebook writes its outputs.
> The service listens on <http://127.0.0.1:8000>, and its interactive documentation is at <http://127.0.0.1:8000/docs>.

## Setup

### _imports_

In [37]:
import json
import time
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import ProxyHandler, Request, build_opener

In [ ]:
import pandas as pd
import plotly.io  # pyright: ignore

### _helpers_

In [ ]:
API = "http://127.0.0.1:8000"
opener = build_opener(ProxyHandler({}))

In [ ]:
def call(method, path, params=None, body=None):  # pyright: ignore
    url = API + path + ("?" + urlencode(params) if params else "")  # pyright: ignore
    data = None if body is None else json.dumps(body).encode()
    request = Request(
        url, data=data, method=method, headers={"Content-Type": "application/json"}  # pyright: ignore
	)
    try:
        with opener.open(request, timeout=30) as response:
            return response.status, json.load(response)
    except HTTPError as error:
        return error.code, json.load(error)


def get(path, **params):  # pyright: ignore
    return call("GET", path, params)  # pyright: ignore


def post(path, body=None):  # pyright: ignore
    return call("POST", path, body=body)  # pyright: ignore

### _checks_

Verify that service answers:

In [52]:
status, openapi = get("/openapi.json")

print(f"{status=}")
print(openapi["info"])

status=200
{'title': 'Central bank policy rates', 'description': 'Policy rates by country or economic area, and the pipeline jobs.', 'version': '0.2.0'}


Verify that something is actually staged:

In [ ]:
status, body = get("/ref-areas")
if status == 200:
    print(f"{len(body)} countries and economic areas staged")
else:
    print(status, body)

## Run the pipeline as jobs

`POST /jobs/fetch` and `POST /jobs/transform` answer at once with `202` and a job.  
The work runs in the background, one job at a time.  


`wait` polls `GET /jobs/{id}` until the job has succeeded or failed.

In [ ]:
def wait(job, timeout=600): # timeout in seconds
    deadline = time.monotonic() + timeout
    while job["status"] in ("queued", "running"):
        if time.monotonic() > deadline:
            raise TimeoutError(f"job {job['id']} is still {job['status']}")
        time.sleep(1)
        _, job = get(f"/jobs/{job['id']}")
    return job

Download the BIS file, or reuse the copy downloaded in the last 24 hours:

In [53]:
status, job = post("/jobs/fetch")
print(f"status={job['status']}")

fetched = wait(job)
print(f"status={fetched['status']}")

status=queued
status=succeeded


In [ ]:
(fetched.get("result", {})).get("release", fetched["error"])

Stage it. After this job, the read endpoints serve the new data without a restart:

In [ ]:
status, job = post("/jobs/transform")
staged = wait(job)
print("status:", staged["status"])

In [34]:
staged.get("result", staged["error"])

{'observations': 709984, 'series': 49, 'findings': 0}

A job started while another runs is refused with `409`, so two jobs never write the same files at once.

## The countries and economic areas

`GET /ref-areas` lists every code the data has, with its name. `EA` is an alias of the BIS code `XM`.

In [40]:
status, ref_areas = get("/ref-areas")
print(f"{status=}")

status=200


In [45]:
df = pd.DataFrame(ref_areas)
df.head(10)

,ref_area,ref_area_name,aliases
0,AR,Argentina,[]
1,AT,Austria,[]
2,AU,Australia,[]
3,BE,Belgium,[]
4,BR,Brazil,[]
5,CA,Canada,[]
6,CH,Switzerland,[]
7,CL,Chile,[]
8,CN,China,[]
9,CO,Colombia,[]


In [46]:
df[df["ref_area_name"].str.contains("Euro", na=False)]

,ref_area,ref_area_name,aliases
47,XM,Euro area,[EA]


## The summary

`GET /summary` gives, for each requested code, the latest rate, the rate 12 months earlier and the change in basis points:  
the same document `bis-prates report` writes to `summary.json`.  

A code the data does not have is not an error: its status is `unknown_ref_area`.

In [47]:
status, document = get("/summary", countries="US,EA,GB,JP,CH,ZZ")
columns = [
    "requested_ref_area", "ref_area", "latest_date", "latest_value",
    "prior_date", "prior_value", "change_bps", "status", "flags",
]
pd.DataFrame(document["snapshots"])[columns]

,requested_ref_area,ref_area,latest_date,latest_value,prior_date,prior_value,change_bps,status,flags
0,US,US,2026-11-17,3.875,2025-11-17,3.875,0.0,ok,[]
1,EA,XM,2026-11-17,2.500,2025-11-17,2.000,50.0,ok,[]
2,GB,GB,2026-11-16,4.000,2025-11-14,4.000,0.0,ok,[]
3,JP,JP,2026-11-17,1.250,2025-11-17,0.500,75.0,ok,[]
4,CH,CH,2026-11-17,0.000,2025-11-17,0.000,0.0,ok,[]
5,ZZ,ZZ,NaN,NaN,NaN,NaN,NaN,unknown_ref_area,[]


In [48]:
document["source"]

{'location': 'https://data.bis.org/static/bulk/WS_CBPOL_csv_flat.zip',
 'version_id': None,
 'checksum_sha256': '79f614ab22c21d172694cd48e9a9d551c7196f53c525008bc7ea4da8831d6494',
 'retrieved_at': '2026-11-21T18:31:09.591146Z'}

## The time series of one country or economic area

`GET /timeseries` gives the valid observations of one code, oldest first, from `start` on.

In [49]:
status, series = get("/timeseries", country="EA", start="2022-01-01")
rates = pd.DataFrame(series["observations"])
print(series["series"]["title"])
rates.tail()

Central bank policy rates - Euro area - Daily - End of period


,date,value
1745,2026-11-13,2.5
1746,2026-11-14,2.5
1747,2026-11-15,2.5
1748,2026-11-16,2.5
1749,2026-11-17,2.5


The euro-area rate on the start date, then every change since:

In [50]:
rates[rates["value"].diff() != 0]

,date,value
0,2022-01-01,0.00
207,2022-07-27,0.50
256,2022-09-14,1.25
305,2022-11-02,2.00
354,2022-12-21,2.50
403,2023-02-08,3.00
445,2023-03-22,3.50
494,2023-05-10,3.75
536,2023-06-21,4.00
578,2023-08-02,4.25


## The chart

`GET /chart` returns a Plotly figure in JSON: the same lines as the report's `policy_rates.png`, as the web front end will draw them.  
Hover over a line to read the date and the rate.

In [51]:
status, figure = get("/chart", countries="US,EA,GB,JP,CH", start="2015-01-01")
plotly.io.from_json(json.dumps(figure))  # pyright: ignore

## The report as a job

`POST /jobs/report` writes the four report files into the service's `--out` folder.

In [54]:
status, job = post(
	"/jobs/report",
	{
		"countries": ["US", "EA", "GB", "JP", "CH"],
		"start": "2015-01-01"
	}
)

report = wait(job)
print(f"status={report['status']}")


status=succeeded


In [56]:
(report.get("result", {})).get("files", report['error'])

['/tmp/bis-prates-out/summary.csv',
 '/tmp/bis-prates-out/summary.json',
 '/tmp/bis-prates-out/policy_rates.png',
 '/tmp/bis-prates-out/report.md']

## Errors

Every error has the same body. A code the data does not have, in `/timeseries`:

In [57]:
get("/timeseries", country="ZZ")

(404,
 {'error': 'not_found',
  'message': 'no series for ZZ; GET /ref-areas lists the codes'})

A request with no code at all:

In [58]:
get("/summary", countries=" , ")

(422,
 {'error': 'invalid_request',
  'message': 'name at least one country or area code, e.g. US or EA'})